# 🧱 Base analítica: do lake à tabela de modelagem

Este notebook monta a **tabela analítica** que alimentará os modelos: cada linha é um aluno avaliado, com a variável resposta (alfabetizado ou não) e o contexto do seu município. É a primeira etapa do projeto e a fundação de todas as seguintes, porque define a unidade de predição, as variáveis disponíveis e a estratégia que impede o vazamento de dados.

> **Convenção de trabalho:** o desenvolvimento acontece neste notebook, célula a célula, com os resultados salvos. Ao final da etapa, o código estável é promovido para `src/preprocessing/prod_01_base_analitica.py`.

**Decisões que governam este notebook** (ver [diário de decisões](../docs/decisoes.md)):
- **D-001, grão de modelagem:** a linha vem da camada Silver (o aluno), o contexto vem da camada Gold e das tabelas municipais (o território), sempre defasado no tempo;
- **D-002, dependência entre as fases:** este projeto verifica o contrato do lake construído na fase anterior, sem reexecutar aquela pipeline.

---

**Estratégia deste notebook:**

```
verificar o       ->  isolar a          ->  montar o          ->  integrar e   ->  desenhar o    ->  gravar a
lake (pré-voo)        população            contexto              auditar          split             base
Seção 1               Seção 2              defasado (3)          Seções 4 e 5     Seção 6           Seção 7
```

O princípio é o mesmo da fase anterior: cada passo é conferido contra um número esperado antes do seguinte, e nada é gravado sem verificação.

## 1. Setup e verificação de pré-requisitos

**Passos desta seção:** (1.1) configurar o acesso ao data lake da fase anterior; (1.2) verificar se as camadas exigidas existem e têm as colunas esperadas.

🎓 **Conceito, contrato entre pipelines:** a pipeline de dados da fase anterior é *upstream*; este projeto de machine learning é *downstream*. Quem está a jusante **verifica** se o insumo existe no formato esperado, e não reexecuta a pipeline de cima: reexecutar acoplaria os dois repositórios e duplicaria responsabilidade. Se algo faltar, a execução para com uma mensagem que orienta o que rodar antes (decisão D-002).

> 📌 **Nota para reprodução:** este notebook lê `config/config.json`, que não é versionado. Crie o seu a partir de `config/config.example.json`, apontando para o projeto e o bucket onde a pipeline da fase anterior foi executada.

In [81]:
# --- 1.1 Configuração e acesso ao data lake ---
import json
from pathlib import Path

import pandas as pd
import pydata_google_auth
from google.cloud import storage

CFG = json.loads(Path("../config/config.json").read_text(encoding="utf-8"))
PROJETO_GCP = CFG["projeto_gcp"]
BUCKET_LAKE = CFG["bucket_lake"]

ESCOPOS = ["https://www.googleapis.com/auth/cloud-platform"]
credenciais = pydata_google_auth.get_user_credentials(ESCOPOS)
credenciais = credenciais.with_quota_project(PROJETO_GCP)
cliente_storage = storage.Client(project=PROJETO_GCP, credentials=credenciais)

def garantir_credencial(forcar: bool = False) -> None:
    """Renova o token de acesso e limpa o cache do gcsfs.

    O token de usuário expira em cerca de uma hora. Em sessões longas de
    notebook, qualquer leitura ou gravação no lake feita depois disso
    falha com erro 401. Esta função deve ser chamada antes de cada acesso;
    o parâmetro forcar permite renovar mesmo quando o token ainda parece
    válido, situação em que o gcsfs pode manter em cache uma instância
    autenticada com o token anterior.
    """
    import google.auth.transport.requests
    import gcsfs
    if forcar or not credenciais.valid:
        credenciais.refresh(google.auth.transport.requests.Request())
        gcsfs.GCSFileSystem.clear_instance_cache()

def ultima_particao(area: str, tabela: str) -> str | None:
    """Partição mais recente de uma tabela do lake, ou None se não existir."""
    particoes = sorted({
        b.name.split("/")[2]
        for b in cliente_storage.list_blobs(BUCKET_LAKE, prefix=f"{area}/{tabela}/")
        if len(b.name.split("/")) > 2
    })
    return particoes[-1] if particoes else None

def ler_lake(area: str, tabela: str, **kwargs) -> pd.DataFrame:
    """Lê a partição mais recente de uma tabela do lake."""
    garantir_credencial()
    particao = ultima_particao(area, tabela)
    if particao is None:
        raise FileNotFoundError(
            f"Tabela '{tabela}' não encontrada em {area}/. "
            "Execute antes a pipeline da fase anterior."
        )
    caminho = f"gs://{BUCKET_LAKE}/{area}/{tabela}/{particao}/{tabela}.parquet"
    return pd.read_parquet(caminho, storage_options={"token": credenciais},
                           **kwargs)

print(f"Lake: gs://{BUCKET_LAKE}/")
print(f"Projeto GCP: {PROJETO_GCP}")
print("Setup ok")

Lake: gs://tech-challenge-fase2-lake-rm373453/
Projeto GCP: tech-challenge-fase2
Setup ok


In [82]:
# --- 1.2 Verificação de pré-requisitos (pré-voo) ---
# Contrato: tabelas e colunas que este projeto exige da fase anterior.
CONTRATO = {
    ("silver", "alunos"): ["ano", "id_municipio", "rede_nome", "presente",
                           "alfabetizado", "proficiencia", "peso_aluno"],
    ("silver", "municipio"): ["ano", "id_municipio", "rede", "nome",
                              "sigla_uf", "nome_regiao",
                              "taxa_alfabetizacao", "media_portugues"],
    ("silver", "metas_municipio"): ["ano_referencia", "id_municipio",
                                    "ano_meta", "meta_taxa"],
    ("gold", "indicador_municipio"): ["ano", "id_municipio", "taxa",
                                      "percentual_participacao",
                                      "meta_taxa", "origem"],
}

print("Verificação de pré-requisitos no lake:")
print()
pendencias = []
for (area, tabela), colunas in CONTRATO.items():
    particao = ultima_particao(area, tabela)
    if particao is None:
        print(f"  {area}/{tabela:<22} AUSENTE")
        pendencias.append(f"{area}/{tabela}")
        continue
    amostra = ler_lake(area, tabela, columns=colunas[:1])
    disponiveis = set(pd.read_parquet(
        f"gs://{BUCKET_LAKE}/{area}/{tabela}/{particao}/{tabela}.parquet",
        storage_options={"token": credenciais}).columns)
    faltantes = [c for c in colunas if c not in disponiveis]
    status = "OK" if not faltantes else f"COLUNAS FALTANTES: {faltantes}"
    if faltantes:
        pendencias.append(f"{area}/{tabela}: {faltantes}")
    print(f"  {area}/{tabela:<22} {particao}  {len(amostra):>9,} linhas  {status}")

print()
if pendencias:
    raise RuntimeError(
        "Pré-requisitos não atendidos: " + "; ".join(pendencias) + ".\n"
        "Execute a pipeline da fase anterior (repositório "
        "Tech_Challenge_RM373453_pipeline_alfabetizacao, passos 4 a 8 do "
        "Como Executar) antes de prosseguir."
    )
print("Contrato atendido: o lake tem o que este projeto precisa.")

Verificação de pré-requisitos no lake:

  silver/alunos                 data_processamento=2026-07-11  3,866,814 linhas  OK
  silver/municipio              data_processamento=2026-07-11     23,995 linhas  OK
  silver/metas_municipio        data_processamento=2026-07-11     74,928 linhas  OK
  gold/indicador_municipio    data_processamento=2026-07-12     11,629 linhas  OK

Contrato atendido: o lake tem o que este projeto precisa.


## 2. População de modelagem e variável resposta

**Passos desta seção:** (2.1) carregar os alunos da camada Silver e isolar a população modelável; (2.2) examinar a variável resposta e seu balanceamento.

🎓 **Conceito, população modelável** (decisão D-001): o modelo aprende com quem tem resultado observado. Alunos ausentes na avaliação não têm proficiência registrada e, portanto, não têm variável resposta: eles permanecem no lake, com a flag de presença criada na fase anterior, mas ficam fora do treinamento. Essa exclusão é uma premissa declarada, não um descarte silencioso, e volta como limitação do projeto: o modelo prevê a alfabetização de quem realiza a prova.

⚠️ **Achado do reconhecimento do lake:** o identificador de aluno se repete entre os ciclos de 2023 e 2024 em 86,7% dos casos, embora a avaliação seja aplicada a coortes diferentes (alunos do 2º ano de cada ano). O identificador é, portanto, uma máscara reutilizada, e não permite acompanhar a mesma criança ao longo do tempo. Isso exclui qualquer variável de trajetória individual e reforça que o identificador não entra no modelo.

In [83]:
# --- 2.1 Carregar alunos e isolar a população modelável ---
df_alunos = ler_lake(
    "silver", "alunos",
    columns=["ano", "id_municipio", "rede_nome", "presente",
             "alfabetizado", "proficiencia", "peso_aluno"],
)
print(f"Alunos na camada Silver: {len(df_alunos):,}")

# A população modelável: presentes na avaliação (D-001)
df_pop = df_alunos[df_alunos["presente"]].copy()
print(f"Presentes (com resultado observado): {len(df_pop):,} "
      f"({len(df_pop) / len(df_alunos):.1%})")
print(f"Ausentes (fora do treinamento):      "
      f"{len(df_alunos) - len(df_pop):,}")
print()

# Variável resposta: alfabetizado (1) ou não (0)
df_pop["alvo"] = (df_pop["alfabetizado"].astype(str) == "1").astype(int)

print("Distribuição por ciclo:")
print(df_pop.groupby("ano")["alvo"]
      .agg(alunos="size", alfabetizados="sum",
           taxa=lambda s: f"{100 * s.mean():.1f}%").to_string())

Alunos na camada Silver: 3,866,814
Presentes (com resultado observado): 3,354,661 (86.8%)
Ausentes (fora do treinamento):      512,153

Distribuição por ciclo:
       alunos  alfabetizados   taxa
ano                                
2023  1502809         877427  58.4%
2024  1851852        1107119  59.8%


In [84]:
# --- 2.2 Conferir a variável resposta contra o dado oficial ---
# A taxa da população modelável é a taxa NÃO ponderada; a oficial usa o
# peso amostral. As duas devem ficar próximas, e a diferença entre elas
# já antecipa a decisão pendente sobre o uso do peso no treinamento.

for ano in sorted(df_pop["ano"].unique()):
    recorte = df_pop[df_pop["ano"] == ano]
    simples = 100 * recorte["alvo"].mean()
    ponderada = (100 * (recorte["peso_aluno"] * recorte["alvo"]).sum()
                 / recorte["peso_aluno"].sum())
    print(f"{ano}:  simples {simples:.1f}%   ponderada {ponderada:.1f}%   "
          f"diferença {ponderada - simples:+.1f} pp")

print()
print("Referência oficial da rede pública (INEP): 55,9% em 2023 e 59,2% em 2024.")
print("A população acima inclui a rede privada, o que eleva ambas as taxas.")
print()
print("Distribuição por rede (todos os ciclos):")
print(df_pop.groupby("rede_nome")["alvo"]
      .agg(alunos="size", taxa=lambda s: f"{100 * s.mean():.1f}%").to_string())

2023:  simples 58.4%   ponderada 57.5%   diferença -0.9 pp
2024:  simples 59.8%   ponderada 59.2%   diferença -0.6 pp

Referência oficial da rede pública (INEP): 55,9% em 2023 e 59,2% em 2024.
A população acima inclui a rede privada, o que eleva ambas as taxas.

Distribuição por rede (todos os ciclos):
            alunos   taxa
rede_nome                
Estadual    372596  62.1%
Municipal  2982041  58.8%
Privada         24  66.7%


### 2.3 Dois achados que orientam as próximas decisões

- **O peso amostral reproduz o número oficial.** A taxa ponderada de 2024 (59,2%) coincide com a taxa divulgada pelo INEP para a rede pública naquele ciclo, enquanto a taxa simples fica 0,6 ponto acima. O peso, calibrado pelo instituto, corrige a sub-representação de certos perfis na amostra: é a evidência que sustentará a decisão sobre usá-lo como ponderação no treinamento.
- **A rede privada é residual nesta base:** 24 alunos entre 3,35 milhões. Não constitui uma categoria com massa estatística; será tratada de forma explícita na preparação, para não gerar uma classe rara sem significado no modelo.

## 3. Contexto defasado: da rede do aluno e do seu município

**Passos desta seção:** (3.1) montar o retrato da rede de ensino de cada município no ciclo anterior; (3.2) acrescentar o retrato do município como um todo e as variáveis conhecidas antes da avaliação.

🎓 **Conceito, o que o modelo pode saber** (*informação ex-ante × ex-post*): a regra que separa uma variável legítima de um vazamento não é o assunto dela, é o **momento em que ela passa a existir**. No instante da predição já se conhece a situação do território no ciclo anterior e a meta pactuada para o ciclo corrente; ainda não se conhece o resultado do próprio ciclo.

| Variável | Quando passa a existir | Entra no modelo |
|---|---|---|
| Desempenho da rede e do município no ciclo **anterior** | antes da avaliação | ✅ sim |
| **Meta** pactuada para o ciclo corrente | antes da avaliação (é um pacto prévio) | ✅ sim |
| Taxa do município no ciclo **corrente** | depois da avaliação, e calculada **com o próprio aluno** | ❌ não |

O caso da meta merece destaque: ela se refere ao ciclo que se quer prever, mas é **conhecida de antemão**, porque foi pactuada entre os entes federativos antes da aplicação da prova. Não é vazamento; é exatamente a informação que um gestor teria em mãos ao tentar antecipar o resultado.

🎓 **Conceito, o grão do contexto:** o aluno pertence a uma **rede** dentro de um **município**, e as duas camadas informam coisas diferentes. A rede estadual e a rede municipal de uma mesma cidade divergem mais do que se imagina: nos 1.083 municípios em que ambas foram medidas em 2023, a diferença entre elas tem desvio padrão de 19,5 pontos percentuais e supera 10 pontos em 58% dos casos. Usar apenas o agregado das duas descartaria essa variação e trataria alunos de realidades distintas como se vivessem a mesma. Por isso o contexto é montado em dois níveis:

- **rede do aluno**, o desempenho da rede que efetivamente o atende;
- **município**, o clima educacional do território como um todo, com participação e porte;
- e a **diferença entre os dois**, que posiciona a rede do aluno acima ou abaixo do seu município.

📌 **Consequência do desenho temporal:** como a camada Silver cobre 2023 e 2024, apenas o ciclo de **2024** reúne aluno e contexto anterior. O ciclo de 2023 **não entra como linha, e sim como contexto**: o retrato daquele ano vira coluna em todas as observações. Isso evita dois erros opostos, o de misturar ciclos deixando metade das linhas sem contexto, e o de usar o contexto do próprio ciclo, que seria vazamento.

In [85]:
# --- 3.1 Retrato da rede de ensino no ciclo anterior ---
CICLO_ALVO = 2024
CICLO_ANTERIOR = CICLO_ALVO - 1

mun = ler_lake("silver", "municipio",
               columns=["ano", "id_municipio", "rede", "rede_nome",
                        "taxa_alfabetizacao", "media_portugues",
                        "sigla_uf", "nome_regiao"])

# Contexto no grão da rede: a rede que atende o aluno (Estadual ou Municipal)
ctx_rede = mun.loc[
    (mun["ano"] == CICLO_ANTERIOR) & (mun["rede"].astype(str).isin(["2", "3"])),
    ["id_municipio", "rede_nome", "taxa_alfabetizacao", "media_portugues"],
].rename(columns={
    "taxa_alfabetizacao": "rede_taxa_ant",
    "media_portugues": "rede_media_portugues_ant",
})

print(f"Retratos de rede em {CICLO_ANTERIOR}: {len(ctx_rede):,}")
print(ctx_rede.groupby("rede_nome", observed=True)
      .agg(municipios=("id_municipio", "nunique"),
           taxa_media=("rede_taxa_ant", lambda s: f"{s.mean():.1f}%"))
      .to_string())
print()

# Quanto as redes divergem dentro do mesmo município
comparativo = ctx_rede.pivot_table(index="id_municipio", columns="rede_nome",
                                   values="rede_taxa_ant", observed=True)
ambas = comparativo.dropna()
diferenca = ambas["Estadual"] - ambas["Municipal"]
print(f"Municípios com as duas redes medidas: {len(ambas):,}")
print(f"Diferença Estadual - Municipal: média {diferenca.mean():+.1f} pp, "
      f"desvio {diferenca.std():.1f} pp")
print(f"Acima de 10 pp de diferença: {100 * (diferenca.abs() > 10).mean():.0f}% "
      f"dos municípios")

Retratos de rede em 2023: 6,597
           municipios taxa_media
rede_nome                       
Estadual         1149      63.8%
Municipal        5448      60.3%

Municípios com as duas redes medidas: 1,083
Diferença Estadual - Municipal: média +3.7 pp, desvio 19.5 pp
Acima de 10 pp de diferença: 58% dos municípios


In [86]:
# --- 3.2 Retrato do município e variáveis conhecidas antes da avaliação ---
# Município como um todo (rede pública), vindo da camada Gold
gold = ler_lake("gold", "indicador_municipio")
ctx_mun = gold.loc[
    (gold["ano"] == CICLO_ANTERIOR) & (gold["origem"] == "oficial_inep"),
    ["id_municipio", "taxa", "percentual_participacao", "taxa_ajustada",
     "alunos_presentes"],
].rename(columns={
    "taxa": "mun_taxa_ant",
    "percentual_participacao": "mun_participacao_ant",
    "taxa_ajustada": "mun_taxa_ajustada_ant",
    "alunos_presentes": "mun_alunos_ant",
})

# Território: unidade da federação e região (estáveis no tempo)
territorio = mun.loc[
    (mun["ano"] == CICLO_ANTERIOR) & (mun["rede"].astype(str) == "5"),
    ["id_municipio", "sigla_uf", "nome_regiao"],
]
ctx_mun = ctx_mun.merge(territorio, on="id_municipio", how="left")

# Meta pactuada para o ciclo alvo: informação ex-ante
metas = gold.loc[
    (gold["ano"] == CICLO_ALVO) & (gold["origem"] == "oficial_inep"),
    ["id_municipio", "meta_taxa"],
].rename(columns={"meta_taxa": "mun_meta_ciclo"})
ctx_mun = ctx_mun.merge(metas, on="id_municipio", how="left")
ctx_mun["mun_gap_meta"] = ctx_mun["mun_meta_ciclo"] - ctx_mun["mun_taxa_ant"]

# Espaço reservado para o enriquecimento externo (etapa 3 do plano):
# fontes municipais novas entram aqui, por join em id_municipio.

print(f"Contexto municipal: {len(ctx_mun):,} municípios, "
      f"{len(ctx_mun.columns) - 1} variáveis")
print()
print("Preenchimento das variáveis municipais:")
print(pd.DataFrame({
    "% preenchido": (100 * ctx_mun.notna().mean()).round(1)
}).drop(index="id_municipio").to_string())

Contexto municipal: 4,950 municípios, 8 variáveis

Preenchimento das variáveis municipais:
                       % preenchido
mun_taxa_ant                  100.0
mun_participacao_ant           98.4
mun_taxa_ajustada_ant          98.4
mun_alunos_ant                 98.4
sigla_uf                      100.0
nome_regiao                   100.0
mun_meta_ciclo                 94.7
mun_gap_meta                   94.7


### 3.3 Benchmark estadual: comparar a rede com os seus pares

🎓 **Conceito, posição relativa ao grupo de pares:** o valor absoluto de um indicador diz pouco sem referência. Uma rede municipal com 60% de alfabetização representa uma situação boa em um estado cuja mediana é 50%, e ruim em outro cuja mediana é 70%. O que informa é a **posição relativa**, e os pares corretos para comparação são as redes do **mesmo tipo**, no **mesmo estado**, porque compartilham política estadual, contexto socioeconômico e regime de colaboração entre os entes.

Duas variáveis nascem daí: o benchmark em si (a mediana da rede na unidade da federação) e a distância da rede do aluno até ele. A mediana é preferida à média por ser robusta a municípios atípicos, comuns em estados com poucas redes medidas.

📌 **Sem vazamento:** o benchmark é calculado sobre o ciclo anterior, o mesmo do restante do contexto. Ele não contém informação do ciclo que se quer prever.

In [87]:
# --- 3.3 Benchmark estadual por rede ---
# Mediana da taxa de cada rede entre os municípios da mesma UF (ciclo anterior)
base_bench = ctx_rede.merge(
    mun.loc[mun["ano"] == CICLO_ANTERIOR, ["id_municipio", "sigla_uf"]]
       .drop_duplicates(),
    on="id_municipio", how="left")

benchmark = (base_bench.groupby(["sigla_uf", "rede_nome"], observed=True)
             ["rede_taxa_ant"].median()
             .rename("uf_rede_taxa_ant").reset_index())

print(f"Benchmarks calculados: {len(benchmark)} combinações de UF e rede")
print()
print("Amostra (as cinco maiores e as cinco menores medianas):")
ordenado = benchmark.sort_values("uf_rede_taxa_ant", ascending=False)
print(pd.concat([ordenado.head(5), ordenado.tail(5)])
      .round(1).to_string(index=False))
print()

# Acoplar o benchmark ao contexto de rede
ctx_rede = base_bench.merge(benchmark, on=["sigla_uf", "rede_nome"], how="left")
ctx_rede["rede_vs_uf"] = (ctx_rede["rede_taxa_ant"]
                          - ctx_rede["uf_rede_taxa_ant"])
ctx_rede = ctx_rede.drop(columns="sigla_uf")

print("Distância da rede até o benchmark do seu estado (pontos percentuais):")
print(ctx_rede["rede_vs_uf"].describe().round(1).to_string())

Benchmarks calculados: 46 combinações de UF e rede

Amostra (as cinco maiores e as cinco menores medianas):
sigla_uf rede_nome  uf_rede_taxa_ant
      CE Municipal              93.2
      PR  Estadual              84.7
      CE  Estadual              84.2
      GO  Estadual              79.5
      ES  Estadual              79.1
      BA Municipal              36.1
      RN Municipal              36.0
      AL  Estadual              30.1
      SE Municipal              28.9
      BA  Estadual              27.5

Distância da rede até o benchmark do seu estado (pontos percentuais):
count    6597.0
mean        0.3
std        15.4
min       -66.4
25%        -9.6
50%         0.0
75%         9.7
max        58.2


### 3.4 Porte do ciclo corrente: o que já se sabe antes da prova

🎓 **Conceito, nem tudo do ciclo corrente é vazamento.** A regra continua sendo o momento em que a informação passa a existir. O **porte** da rede e do município no ciclo que se quer prever, isto é, quantos alunos há para avaliar, vem do cadastro escolar e está definido **antes** da aplicação da prova. Não depende de nenhum resultado, e por isso é informação legítima.

⚠️ **A linha fina:** alunos **avaliáveis** (matriculados, presentes e ausentes) é informação prévia; alunos **presentes** só se conhece depois da aplicação, e seria vazamento. O porte aqui conta os avaliáveis.

📌 **Por que preferir o porte corrente ao do ciclo anterior:** o porte defasado sofre de dois problemas. Envelhece (redes crescem e encolhem entre ciclos) e depende da disponibilidade dos microdados do ano anterior, o que restringe sua cobertura a 76,9% das observações. O porte corrente é calculado da própria base de alunos e cobre a totalidade. Não à toa, o porte defasado foi a variável com a menor correlação com a resposta entre todas as testadas.

Da mesma estrutura nasce uma terceira variável, que nenhuma outra expressa: a **fração do município atendida pela rede do aluno**. Ela distingue quem estuda na rede predominante do território de quem está em uma rede minoritária ali.

In [88]:
# --- 3.4 Porte do ciclo corrente (informação prévia à avaliação) ---
# Base completa do ciclo (presentes e ausentes): é o cadastro de avaliáveis
cadastro = df_alunos[df_alunos["ano"] == CICLO_ALVO]

porte_rede = (cadastro.groupby(["id_municipio", "rede_nome"], observed=True)
              .size().rename("rede_porte_atual").reset_index())
porte_mun = (cadastro.groupby("id_municipio", observed=True)
             .size().rename("mun_porte_atual").reset_index())

print(f"Avaliáveis no ciclo {CICLO_ALVO}: {len(cadastro):,} "
      f"(presentes e ausentes)")
print(f"Combinações município e rede: {len(porte_rede):,}")
print(f"Municípios: {len(porte_mun):,}")
print()
print("Porte por rede (alunos avaliáveis por município):")
print(porte_rede.groupby("rede_nome", observed=True)["rede_porte_atual"]
      .describe()[["count", "mean", "50%", "max"]].round(0).to_string())

Avaliáveis no ciclo 2024: 2,119,624 (presentes e ausentes)
Combinações município e rede: 6,543
Municípios: 5,519

Porte por rede (alunos avaliáveis por município):
            count   mean    50%      max
rede_nome                               
Estadual   1090.0  257.0   49.0  58612.0
Municipal  5452.0  337.0  114.0  49820.0
Privada       1.0   25.0   25.0     25.0


## 4. Enriquecimento com fontes externas

**Passos desta seção:** (4.1) preparar o acesso às fontes públicas e o cache no lake; (4.2) trazer o retrato da infraestrutura escolar; (4.3) o tamanho de turma da série avaliada; (4.4) o contexto econômico do município; (4.5) o contexto socioeconômico estrutural; (4.6) a exposição à violência.

🎓 **Por que esta seção existe.** As variáveis construídas até aqui descrevem o **desempenho anterior** do território: taxa da rede, taxa do município, meta pactuada, distância até o benchmark. São informativas, mas todas medem a mesma coisa que se quer prever, apenas deslocada no tempo. Um modelo construído só com elas aprende que **quem ia mal continua indo mal**, o que é verdadeiro e inútil: não identifica *fatores*, e não sugere ação nenhuma ao gestor, que não pode intervir sobre a taxa do ano passado.

O enunciado pede outra coisa. Ele pergunta **quais fatores mais impactam a alfabetização** e espera inteligência aplicável a políticas públicas. Isso exige variáveis de natureza diferente, que descrevam as **condições** em que o ensino acontece: a estrutura da escola, a distância que o aluno percorre, o tamanho da turma, a presença de profissionais de apoio, a riqueza e a vulnerabilidade do território. É sobre essas que uma política pode agir.

📌 **Todas as fontes respeitam a regra temporal.** Cada uma entra com informação anterior ao ciclo avaliado, e o critério de disponibilidade considera também o calendário de publicação: o Censo Escolar de 2023 é divulgado antes da avaliação de 2024, enquanto o PIB municipal tem cerca de dois anos de defasagem, e por isso entra o de 2021.

| Fonte | Referência | O que traz | Ressalva |
|---|---|---|---|
| Censo Escolar (INEP) | 2023 | infraestrutura, localização rural, transporte escolar, profissionais de apoio, salas | nenhuma |
| Censo Escolar, turmas | 2023 | tamanho médio da turma do 2º ano | nenhuma |
| PIB municipal (IBGE) | 2021 | PIB per capita e perfil setorial | defasagem de publicação |
| Atlas do Desenvolvimento Humano | 2010 | IDHM, renda, desigualdade, analfabetismo adulto | dado censitário, defasado |
| Atlas da Violência (IPEA) | 2010 | vulnerabilidade social e seus componentes | dado censitário, defasado |
| Mortalidade (SIM/DataSUS) | 2019 | óbitos por agressão, como medida de violência | defasado |

📌 **Sobre o caminho que estes dados percorrem** (decisão D-005): as fontes externas são consultadas diretamente na origem, e o resultado agregado é materializado no data lake para reutilização. Elas não passam pelas camadas do medalhão nesta fase, porque a etapa é de prototipação: o propósito é descobrir **se** essas variáveis carregam informação útil, e industrializar a ingestão de cinco fontes antes dessa resposta seria construir infraestrutura para dados que podem ser descartados na seleção de variáveis. Fica registrada a recomendação de que as fontes aprovadas pela análise de importância sejam incorporadas ao lake pelo time de engenharia de dados, com o dado bruto na camada Bronze e a agregação como transformação explícita na Silver.

⚠️ **Sobre as fontes defasadas.** IDHM, vulnerabilidade social e violência são características **estruturais** do território, que se alteram lentamente. Entram como aproximação do contexto, com a defasagem declarada aqui e nas limitações do projeto. O julgamento sobre sua utilidade não é feito por opinião: se não carregarem informação, a análise de importância dos modelos mostrará isso, e a ausência de efeito também será um achado.

In [89]:
# --- 4.1 Acesso às fontes públicas, com cache no lake ---
import pandas_gbq

def obter_fonte_externa(nome: str, consulta: str,
                        forcar: bool = False) -> pd.DataFrame:
    """Consulta uma fonte pública e guarda o resultado no lake.

    A consulta ao BigQuery é feita uma única vez: o resultado agregado é
    gravado em ml/externas/ e reutilizado nas execuções seguintes. Isso
    torna a construção da base reprodutível sem repetir o custo de leitura
    e sem depender da disponibilidade da fonte a cada execução.
    """
    caminho_blob = f"ml/externas/{nome}/{nome}.parquet"
    blob = cliente_storage.bucket(BUCKET_LAKE).blob(caminho_blob)

    if blob.exists() and not forcar:
        import io
        dados = pd.read_parquet(io.BytesIO(blob.download_as_bytes()))
        print(f"  {nome:<22} {len(dados):>7,} linhas  (cache no lake)")
        return dados

    dados = pandas_gbq.read_gbq(consulta, project_id=PROJETO_GCP,
                                credentials=credenciais,
                                progress_bar_type=None)
    dados.to_parquet(f"gs://{BUCKET_LAKE}/{caminho_blob}", index=False,
                     storage_options={"token": credenciais})
    print(f"  {nome:<22} {len(dados):>7,} linhas  (consultado e gravado)")
    return dados

print("Fontes externas (cache em ml/externas/):")

Fontes externas (cache em ml/externas/):


In [90]:
# --- 4.2 Censo Escolar: a infraestrutura em que o ensino acontece ---
# Códigos de rede na tabela de escolas: 2 estadual, 3 municipal.
CENSO_ESCOLAR = f"""
SELECT id_municipio, rede,
       COUNT(*) AS esc_quantidade,
       ROUND(100 * AVG(IF(tipo_localizacao = 'Rural', 1, 0)), 1) AS esc_pct_rural,
       ROUND(100 * AVG(COALESCE(agua_rede_publica, 0)), 1) AS esc_pct_agua_rede,
       ROUND(100 * AVG(COALESCE(esgoto_rede_publica, 0)), 1) AS esc_pct_esgoto_rede,
       ROUND(100 * AVG(COALESCE(energia_rede_publica, 0)), 1) AS esc_pct_energia_rede,
       ROUND(100 * AVG(COALESCE(internet, 0)), 1) AS esc_pct_internet,
       ROUND(100 * AVG(COALESCE(biblioteca, 0)), 1) AS esc_pct_biblioteca,
       ROUND(100 * AVG(COALESCE(laboratorio_informatica, 0)), 1) AS esc_pct_lab_informatica,
       ROUND(100 * AVG(COALESCE(quadra_esportes, 0)), 1) AS esc_pct_quadra,
       ROUND(100 * AVG(COALESCE(alimentacao, 0)), 1) AS esc_pct_alimentacao,
       ROUND(100 * AVG(COALESCE(profissional_coordenador, 0)), 1) AS esc_pct_coordenador,
       ROUND(100 * AVG(COALESCE(profissional_psicologo, 0)), 1) AS esc_pct_psicologo,
       ROUND(100 * AVG(COALESCE(profissional_assistente_social, 0)), 1) AS esc_pct_assistente_social,
       SUM(quantidade_sala_utilizada) AS esc_salas,
       SUM(quantidade_matricula_utiliza_transporte_publico) AS esc_alunos_transporte
FROM `basedosdados.br_inep_censo_escolar.escola`
WHERE ano = {CICLO_ANTERIOR} AND rede IN ('2', '3')
GROUP BY id_municipio, rede
"""

censo = obter_fonte_externa("censo_escolar", CENSO_ESCOLAR)

# A rede aqui vem em código; o restante da base usa o nome
censo["rede_nome"] = censo["rede"].map({"2": "Estadual", "3": "Municipal"})
censo = censo.drop(columns="rede")

print()
print(f"Municípios cobertos: {censo['id_municipio'].nunique():,}")
print()
print("Perfil médio das redes (não ponderado por porte):")
print(censo.groupby("rede_nome", observed=True)[
    ["esc_pct_rural", "esc_pct_internet", "esc_pct_biblioteca",
     "esc_pct_coordenador", "esc_pct_psicologo"]].mean().round(1).to_string())

  censo_escolar           11,132 linhas  (cache no lake)

Municípios cobertos: 5,570

Perfil médio das redes (não ponderado por porte):
           esc_pct_rural  esc_pct_internet  esc_pct_biblioteca  esc_pct_coordenador  esc_pct_psicologo
rede_nome                                                                                             
Estadual             0.0              89.2                64.9                 22.9                3.1
Municipal            0.0              77.7                24.9                 20.1               21.8


In [91]:
# --- 4.3 Tamanho da turma na série avaliada ---
# Atenção: nesta tabela a rede vem por extenso, e não em código.
# A etapa 15 corresponde ao 2º ano do ensino fundamental de nove anos,
# exatamente a série submetida à avaliação.
TURMAS = f"""
SELECT id_municipio,
       CASE rede WHEN 'estadual' THEN 'Estadual'
                 WHEN 'municipal' THEN 'Municipal' END AS rede_nome,
       ROUND(AVG(quantidade_matriculas), 1) AS turma_media_alunos,
       COUNT(*) AS turma_quantidade
FROM `basedosdados.br_inep_censo_escolar.turma`
WHERE ano = {CICLO_ANTERIOR}
  AND rede IN ('estadual', 'municipal')
  AND etapa_ensino = '15'
GROUP BY id_municipio, rede_nome
"""

turmas = obter_fonte_externa("turmas_2ano", TURMAS)
print()
print("Tamanho médio da turma do 2º ano, por rede:")
print(turmas.groupby("rede_nome", observed=True)["turma_media_alunos"]
      .describe()[["count", "mean", "50%", "max"]].round(1).to_string())

  turmas_2ano              6,847 linhas  (cache no lake)

Tamanho médio da turma do 2º ano, por rede:
            count  mean   50%   max
rede_nome                          
Estadual   1326.0  18.4  19.0  50.0
Municipal  5521.0  19.0  19.1  34.0


In [92]:
# --- 4.4 Contexto econômico do município ---
# O PIB municipal é publicado com cerca de dois anos de defasagem: o ano de
# referência usado é o mais recente disponível antes da avaliação.
ANO_PIB = CICLO_ALVO - 3
ECONOMIA = f"""
SELECT p.id_municipio,
       ROUND(p.pib / NULLIF(pop.populacao, 0), 0) AS mun_pib_per_capita,
       ROUND(100 * p.va_agropecuaria / NULLIF(p.va, 0), 1) AS mun_pct_agropecuaria,
       ROUND(100 * p.va_servicos / NULLIF(p.va, 0), 1) AS mun_pct_servicos,
       pop.populacao AS mun_populacao
FROM `basedosdados.br_ibge_pib.municipio` p
JOIN `basedosdados.br_ibge_populacao.municipio` pop
  ON p.id_municipio = pop.id_municipio AND p.ano = pop.ano
WHERE p.ano = {ANO_PIB}
"""

economia = obter_fonte_externa("economia_municipal", ECONOMIA)
print()
print(f"Referência: {ANO_PIB} (o PIB municipal é divulgado com defasagem)")
print(economia[["mun_pib_per_capita", "mun_pct_agropecuaria",
                "mun_populacao"]].describe(
    percentiles=[.25, .5, .75]).round(0).to_string())

  economia_municipal       5,570 linhas  (cache no lake)

Referência: 2021 (o PIB municipal é divulgado com defasagem)
       mun_pib_per_capita  mun_pct_agropecuaria  mun_populacao
count              5570.0                5570.0         5570.0
mean              33873.0                  24.0        38298.0
std               41909.0                  19.0       224288.0
min                5406.0                 -42.0          771.0
25%               12832.0                   9.0         5454.0
50%               23373.0                  19.0        11732.0
75%               40810.0                  36.0        25765.0
max              920828.0                  94.0     12396372.0


In [93]:
# --- 4.5 Contexto socioeconômico estrutural ---
# Fontes censitárias, com referência em 2010. Entram como aproximação de
# características que mudam devagar, com a defasagem declarada.
DESENVOLVIMENTO = """
SELECT a.id_municipio,
       a.idhm AS mun_idhm,
       a.idhm_e AS mun_idhm_educacao,
       a.idhm_r AS mun_idhm_renda,
       a.renda_pc AS mun_renda_per_capita,
       a.indice_gini AS mun_gini,
       a.taxa_analfabetismo_18_mais AS mun_analfabetismo_adulto,
       a.expectativa_anos_estudo AS mun_expectativa_estudo,
       v.ivs AS mun_ivs,
       v.ivs_infraestrutura_urbana AS mun_ivs_infraestrutura,
       v.ivs_capital_humano AS mun_ivs_capital_humano
FROM `basedosdados.mundo_onu_adh.municipio` a
LEFT JOIN (
    SELECT id_municipio, AVG(ivs) AS ivs,
           AVG(ivs_infraestrutura_urbana) AS ivs_infraestrutura_urbana,
           AVG(ivs_capital_humano) AS ivs_capital_humano
    FROM `basedosdados.br_ipea_avs.municipio`
    WHERE ano = 2010 GROUP BY id_municipio
) v ON a.id_municipio = v.id_municipio
WHERE a.ano = 2010
"""

desenvolvimento = obter_fonte_externa("desenvolvimento_humano", DESENVOLVIMENTO)
print()
print("Indicadores estruturais (referência 2010):")
print(desenvolvimento[["mun_idhm", "mun_renda_per_capita", "mun_gini",
                       "mun_analfabetismo_adulto", "mun_ivs"]]
      .describe(percentiles=[.25, .5, .75]).round(2).to_string())

  desenvolvimento_humano   5,565 linhas  (cache no lake)

Indicadores estruturais (referência 2010):
       mun_idhm  mun_renda_per_capita  mun_gini  mun_analfabetismo_adulto  mun_ivs
count   5565.00               5565.00   5565.00                   5565.00  5565.00
mean       0.66                493.61      0.49                     17.40     0.33
std        0.07                243.27      0.07                     10.70     0.11
min        0.42                 96.25      0.28                      0.97     0.08
25%        0.60                281.12      0.45                      8.59     0.24
50%        0.66                467.65      0.49                     14.11     0.32
75%        0.72                650.62      0.54                     26.26     0.41
max        0.86               2043.74      0.80                     47.64     0.76


In [94]:
# --- 4.6 Exposição à violência ---
# Óbitos por agressão (CID-10 X85 a Y09) no Sistema de Informação sobre
# Mortalidade, convertidos em taxa por cem mil habitantes.
ANO_VIOLENCIA = 2019
VIOLENCIA = f"""
SELECT id_municipio, SUM(numero_obitos) AS obitos_agressao
FROM `basedosdados.br_ms_sim.municipio_causa`
WHERE ano = {ANO_VIOLENCIA}
  AND REGEXP_CONTAINS(causa_basica, r'^(X8[5-9]|X9[0-9]|Y0[0-9])')
GROUP BY id_municipio
"""

violencia = obter_fonte_externa("violencia_municipal", VIOLENCIA)

# Municípios sem registro não são desconhecidos: não houve óbito por agressão
violencia = economia[["id_municipio", "mun_populacao"]].merge(
    violencia, on="id_municipio", how="left")
violencia["obitos_agressao"] = violencia["obitos_agressao"].fillna(0)
violencia["mun_taxa_homicidio"] = (100_000 * violencia["obitos_agressao"]
                                   / violencia["mun_populacao"]).round(1)
violencia = violencia[["id_municipio", "mun_taxa_homicidio"]]

print()
print(f"Taxa de homicídio por cem mil habitantes ({ANO_VIOLENCIA}):")
print(violencia["mun_taxa_homicidio"].describe(
    percentiles=[.25, .5, .75, .95]).round(1).to_string())
print()
print(f"Municípios sem óbito por agressão registrado: "
      f"{(violencia['mun_taxa_homicidio'] == 0).sum():,} "
      f"({(violencia['mun_taxa_homicidio'] == 0).mean():.1%})")

  violencia_municipal      3,887 linhas  (cache no lake)

Taxa de homicídio por cem mil habitantes (2019):
count    5570.0
mean       18.1
std        20.4
min         0.0
25%         0.0
50%        13.2
75%        27.4
95%        55.5
max       193.9

Municípios sem óbito por agressão registrado: 1,683 (30.2%)


## 5. Integração e auditoria contra vazamento

**Passos desta seção:** (5.1) integrar cada aluno ao contexto da sua rede e do seu município, conferindo a cobertura; (4.2) submeter a tabela a uma auditoria explícita contra vazamento.

🎓 **Conceito, vazamento de dados** (*data leakage*): ocorre quando uma variável explicativa carrega informação que, no momento real da predição, ainda não existiria. O modelo aprende um atalho, exibe métricas excelentes na avaliação e falha no uso real. O enunciado exige o tratamento desse problema, e aqui ele tem **três fontes distintas**, cada uma com sua defesa:

| Fonte de vazamento | Como se manifestaria aqui | Defesa adotada |
|---|---|---|
| **Temporal** | usar o desempenho do território no próprio ciclo do aluno | contexto sempre defasado (seção 3) |
| **Da variável resposta** | usar a proficiência do aluno, da qual a resposta é derivada | exclusão explícita, auditada na célula 5.2 |
| **Entre partições** | alunos do mesmo município em treino e em teste | separação por município (seção 6) |

⚠️ **A armadilha mais perigosa é a segunda.** A variável resposta é derivada da proficiência (alfabetizado quando a nota atinge 743 pontos). Se a proficiência entrar como variável explicativa, o modelo atinge acurácia praticamente perfeita sem aprender nada: é o gabarito dentro da prova. Por isso a auditoria mantém uma lista de colunas proibidas e verifica, em código, que nenhuma delas alcançou a tabela final.

In [95]:
# --- 5.1 Integrar aluno, rede, município e contexto externo ---
alunos_ciclo = df_pop[df_pop["ano"] == CICLO_ALVO].copy()
print(f"Alunos presentes em {CICLO_ALVO}: {len(alunos_ciclo):,}")
antes = len(alunos_ciclo)

# Cada fonte entra pela chave do seu próprio grão
FONTES = [
    ("contexto da rede", ctx_rede, ["id_municipio", "rede_nome"]),
    ("contexto do município", ctx_mun, ["id_municipio"]),
    ("porte da rede", porte_rede, ["id_municipio", "rede_nome"]),
    ("porte do município", porte_mun, ["id_municipio"]),
    ("censo escolar", censo, ["id_municipio", "rede_nome"]),
    ("turmas do 2º ano", turmas, ["id_municipio", "rede_nome"]),
    ("economia municipal", economia, ["id_municipio"]),
    ("desenvolvimento humano", desenvolvimento, ["id_municipio"]),
    ("violência", violencia, ["id_municipio"]),
]

abt = alunos_ciclo
for rotulo, tabela, chaves in FONTES:
    abt = abt.merge(tabela, on=chaves, how="left")
    if len(abt) != antes:
        raise RuntimeError(f"O join com '{rotulo}' alterou a contagem: "
                           f"{antes:,} -> {len(abt):,}")

print(f"Linhas após {len(FONTES)} integrações: {len(abt):,}  (esperado: igual)")
print()

# Derivadas que só existem depois de reunir as fontes
abt["rede_vs_municipio"] = abt["rede_taxa_ant"] - abt["mun_taxa_ant"]
abt["rede_peso_no_municipio"] = (abt["rede_porte_atual"]
                                 / abt["mun_porte_atual"])
# Densidade física: salas disponíveis por aluno da rede no município
abt["esc_salas_por_aluno"] = abt["esc_salas"] / abt["rede_porte_atual"]
# Dependência de transporte: proxy da distância entre aluno e escola
abt["esc_pct_transporte"] = (100 * abt["esc_alunos_transporte"]
                             / abt["rede_porte_atual"]).clip(upper=100)
abt = abt.drop(columns=["esc_salas", "esc_alunos_transporte",
                        "turma_quantidade"])

print("Cobertura de cada bloco de variáveis:")
BLOCOS = {
    "desempenho anterior da rede": "rede_taxa_ant",
    "desempenho anterior do município": "mun_taxa_ant",
    "infraestrutura escolar": "esc_pct_internet",
    "tamanho de turma": "turma_media_alunos",
    "economia municipal": "mun_pib_per_capita",
    "desenvolvimento humano": "mun_idhm",
    "violência": "mun_taxa_homicidio",
}
for rotulo, coluna in BLOCOS.items():
    cobertura = abt[coluna].notna()
    print(f"  {rotulo:<34} {cobertura.sum():>9,} alunos  "
          f"({cobertura.mean():.1%})")

Alunos presentes em 2024: 1,851,852
Linhas após 9 integrações: 1,851,852  (esperado: igual)

Cobertura de cada bloco de variáveis:
  desempenho anterior da rede        1,816,270 alunos  (98.1%)
  desempenho anterior do município   1,652,243 alunos  (89.2%)
  infraestrutura escolar             1,851,828 alunos  (100.0%)
  tamanho de turma                   1,851,460 alunos  (100.0%)
  economia municipal                 1,851,852 alunos  (100.0%)
  desenvolvimento humano             1,851,210 alunos  (100.0%)
  violência                          1,851,852 alunos  (100.0%)


In [96]:
# --- 5.2 Auditoria contra vazamento ---
PROIBIDAS = {
    "proficiencia": "a variável resposta é derivada dela (nota >= 743)",
    "alfabetizado": "é a própria variável resposta, em outro formato",
    "presente": "constante na população modelada (todos são presentes)",
    "peso_aluno": "metadado amostral; reservado à ponderação, não é atributo",
}

CHAVES = ["ano", "id_municipio"]
RESPOSTA = "alvo"

# As variáveis explicativas, organizadas pela natureza do que descrevem
FAMILIAS = {
    "aluno": ["rede_nome"],
    "desempenho anterior": [
        "rede_taxa_ant", "rede_media_portugues_ant", "uf_rede_taxa_ant",
        "rede_vs_uf", "rede_vs_municipio", "mun_taxa_ant",
        "mun_participacao_ant", "mun_taxa_ajustada_ant", "mun_alunos_ant",
        "mun_meta_ciclo", "mun_gap_meta"],
    "território": ["sigla_uf", "nome_regiao"],
    "porte e oferta": [
        "rede_porte_atual", "mun_porte_atual", "rede_peso_no_municipio",
        "esc_quantidade", "esc_salas_por_aluno", "turma_media_alunos"],
    "infraestrutura escolar": [
        "esc_pct_rural", "esc_pct_agua_rede", "esc_pct_esgoto_rede",
        "esc_pct_energia_rede", "esc_pct_internet", "esc_pct_biblioteca",
        "esc_pct_lab_informatica", "esc_pct_quadra", "esc_pct_alimentacao",
        "esc_pct_transporte"],
    "profissionais de apoio": [
        "esc_pct_coordenador", "esc_pct_psicologo",
        "esc_pct_assistente_social"],
    "contexto socioeconômico": [
        "mun_pib_per_capita", "mun_pct_agropecuaria", "mun_pct_servicos",
        "mun_populacao", "mun_idhm", "mun_idhm_educacao", "mun_idhm_renda",
        "mun_renda_per_capita", "mun_gini", "mun_analfabetismo_adulto",
        "mun_expectativa_estudo", "mun_ivs", "mun_ivs_infraestrutura",
        "mun_ivs_capital_humano", "mun_taxa_homicidio"],
}
FEATURES = [v for grupo in FAMILIAS.values() for v in grupo]

print("Auditoria contra vazamento")
print("=" * 62)

ausentes = [v for v in FEATURES if v not in abt.columns]
if ausentes:
    raise RuntimeError(f"Variáveis declaradas mas ausentes na tabela: {ausentes}")

invasoras = [c for c in FEATURES if c in PROIBIDAS]
print(f"1. Colunas proibidas entre as variáveis: {invasoras or 'nenhuma'}")
for coluna, motivo in PROIBIDAS.items():
    print(f"     {coluna:<14} fora do modelo, {motivo}")

# Toda variável do ciclo corrente precisa de justificativa prévia à prova
EX_ANTE_DO_CICLO = {
    "rede_porte_atual", "mun_porte_atual", "rede_peso_no_municipio",
    "mun_meta_ciclo", "mun_gap_meta"}
DERIVADAS = {"rede_vs_municipio", "rede_vs_uf", "esc_salas_por_aluno",
             "esc_pct_transporte"}
CATEGORICAS_OK = {"rede_nome", "sigla_uf", "nome_regiao"}
suspeitas = [c for c in FEATURES
             if not c.endswith("_ant") and not c.startswith(("esc_", "mun_", "turma_", "uf_"))
             and c not in EX_ANTE_DO_CICLO | DERIVADAS | CATEGORICAS_OK]
print(f"2. Variáveis sem justificativa temporal: {suspeitas or 'nenhuma'}")
print("     admitidas do ciclo corrente: meta pactuada e porte do cadastro")
print(f"     fontes externas: referências anteriores ao ciclo "
      f"({CICLO_ANTERIOR} para o censo escolar, {ANO_PIB} para a economia, "
      f"2010 e {ANO_VIOLENCIA} para as estruturais)")

numericas = [c for c in FEATURES if pd.api.types.is_numeric_dtype(abt[c])]
correl = abt[numericas + [RESPOSTA]].corr()[RESPOSTA].drop(RESPOSTA)
print(f"3. Correlação com a resposta: {len(numericas)} variáveis numéricas")
print("   as dez mais associadas:")
print(correl.reindex(correl.abs().sort_values(ascending=False).index)
      .head(10).round(3).to_string())
suspeita_alta = correl[correl.abs() > 0.9]
print(f"   correlações acima de 0,9: {list(suspeita_alta.index) or 'nenhuma'}")

print("=" * 62)
if invasoras or suspeitas or len(suspeita_alta):
    raise RuntimeError("Auditoria reprovada: revise as variáveis acima.")
print(f"Auditoria aprovada. {len(FEATURES)} variáveis explicativas em "
      f"{len(FAMILIAS)} famílias, {len(abt):,} observações.")
for familia, variaveis in FAMILIAS.items():
    print(f"     {familia:<26} {len(variaveis):>2} variáveis")

Auditoria contra vazamento
1. Colunas proibidas entre as variáveis: nenhuma
     proficiencia   fora do modelo, a variável resposta é derivada dela (nota >= 743)
     alfabetizado   fora do modelo, é a própria variável resposta, em outro formato
     presente       fora do modelo, constante na população modelada (todos são presentes)
     peso_aluno     fora do modelo, metadado amostral; reservado à ponderação, não é atributo
2. Variáveis sem justificativa temporal: nenhuma
     admitidas do ciclo corrente: meta pactuada e porte do cadastro
     fontes externas: referências anteriores ao ciclo (2023 para o censo escolar, 2021 para a economia, 2010 e 2019 para as estruturais)
3. Correlação com a resposta: 45 variáveis numéricas
   as dez mais associadas:
mun_taxa_ajustada_ant       0.275
mun_taxa_ant                0.248
rede_media_portugues_ant    0.247
rede_taxa_ant               0.246
mun_meta_ciclo              0.232
uf_rede_taxa_ant            0.192
mun_gap_meta               -0.18

### 5.3 O que a integração revelou

O contexto no grão da rede cobre mais alunos do que o agregado municipal cobria, porque a maior parte da população estuda na rede municipal, que tem presença em praticamente todos os municípios avaliados. As lacunas remanescentes concentram-se em dois grupos: alunos da rede estadual em municípios cuja rede estadual não foi medida no ciclo anterior, e alunos da rede privada, residual nesta base e sem contexto correspondente.

As variáveis derivadas dos microdados no nível do município (participação, taxa ajustada e total de alunos presentes) permanecem com preenchimento menor, por causa dos municípios que constam no indicador consolidado sem microdados públicos, um ponto cego identificado na fase anterior. Como vários deles são populosos, o efeito sobre a contagem de alunos é maior do que sobre a de municípios.

O tratamento desses ausentes é assunto da etapa de pré-processamento, com duas alternativas a comparar: imputação por medida de tendência central dentro do próprio estado, ou descarte das variáveis que a análise exploratória mostrar pouco informativas. A decisão será registrada no diário com a evidência que a sustentar.

## 6. Partição dos dados: por município, não por sorteio

**Passos desta seção:** (6.1) separar treino, validação e teste por município; (6.2) conferir que a partição é honesta e equilibrada.

🎓 **Conceito, por que não o `train_test_split`:** a função mais conhecida do scikit-learn sorteia **linhas** ao acaso, e é a escolha correta quando cada observação é independente das demais. Não é o caso aqui: os alunos estão **aninhados em municípios**, e nove das dez variáveis explicativas são municipais, idênticas para todos os alunos de um mesmo território.

O efeito fica claro com um exemplo desta base. São Paulo contribui com cerca de cem mil alunos, todos com os mesmos valores de contexto. Num sorteio por linha, parte deles ficaria no treino e parte no teste; o modelo aprenderia a associação entre aquele conjunto específico de valores e o resultado, e voltaria a encontrá-lo na avaliação. A métrica subiria sem que houvesse generalização: o modelo teria memorizado o município, não aprendido o fenômeno. O efeito tem nome, **vazamento por agrupamento**, e é a terceira fonte listada na seção 5.

A resposta do próprio scikit-learn para dados agrupados é o `GroupShuffleSplit`, do mesmo módulo `model_selection`: em vez de sortear linhas, ele sorteia **grupos**, mantendo todas as observações de um município do mesmo lado da fronteira. O critério de escolha entre as duas funções é a estrutura do dado, não o costume:

| Estrutura das observações | Função adequada |
|---|---|
| Independentes (uma linha, uma entidade) | `train_test_split` |
| Aninhadas em grupos (alunos em municípios, consultas em pacientes) | `GroupShuffleSplit`, `GroupKFold` |

A partição por município resolve isso e, mais do que resolver, **mede o que interessa**: o conjunto de teste passa a ser formado por municípios que o modelo nunca viu, que é a situação real de uso, quando se pretende antecipar o risco de territórios ainda não avaliados no ciclo.

📌 **Proporção adotada:** 60% dos municípios para treino, 20% para validação e 20% para teste. A validação serve à escolha de modelos e ao ajuste de hiperparâmetros; o teste permanece intocado até a avaliação final, para preservar a honestidade da estimativa de desempenho.

In [97]:
# --- 6.1 Separar treino, validação e teste por município ---
from sklearn.model_selection import GroupShuffleSplit

SEMENTE = 42  # replicabilidade: mesma semente, mesma partição

municipios = abt["id_municipio"]

# Primeira divisão: 60% treino, 40% para dividir entre validação e teste
divisor = GroupShuffleSplit(n_splits=1, train_size=0.6, random_state=SEMENTE)
idx_treino, idx_resto = next(divisor.split(abt, groups=municipios))

# Segunda divisão: metade do resto para validação, metade para teste
resto = abt.iloc[idx_resto]
divisor2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEMENTE)
idx_val, idx_teste = next(divisor2.split(resto, groups=resto["id_municipio"]))

abt["particao"] = "treino"
abt.iloc[idx_resto, abt.columns.get_loc("particao")] = "teste"
abt.iloc[idx_resto[idx_val], abt.columns.get_loc("particao")] = "validacao"

print("Distribuição das partições:")
print(abt.groupby("particao")
      .agg(alunos=("alvo", "size"),
           municipios=("id_municipio", "nunique"),
           taxa_alfabetizacao=("alvo", lambda s: f"{100 * s.mean():.1f}%"))
      .to_string())

Distribuição das partições:
            alunos  municipios taxa_alfabetizacao
particao                                         
teste       311794        1104              60.8%
treino     1156013        3310              59.4%
validacao   384045        1103              60.0%


In [98]:
# --- 6.2 Conferir que a partição é honesta ---
print("Verificações da partição")
print("=" * 58)

# 1. Nenhum município aparece em mais de uma partição
por_municipio = abt.groupby("id_municipio")["particao"].nunique()
vazados = (por_municipio > 1).sum()
print(f"1. Municípios em mais de uma partição: {vazados}  (esperado: 0)")

# 2. Todas as observações foram atribuídas
sem_particao = abt["particao"].isna().sum()
print(f"2. Observações sem partição: {sem_particao}  (esperado: 0)")

# 3. A variável resposta tem distribuição semelhante entre as partições
taxas = abt.groupby("particao")["alvo"].mean() * 100
amplitude = taxas.max() - taxas.min()
print(f"3. Taxa de alfabetização por partição: "
      f"{', '.join(f'{p} {t:.1f}%' for p, t in taxas.items())}")
print(f"   amplitude entre partições: {amplitude:.1f} pp "
      f"(diferenças pequenas são esperadas: a partição é por município)")

# 4. A cobertura de contexto é semelhante entre as partições
cobertura = abt.groupby("particao")["mun_taxa_ant"].apply(
    lambda s: 100 * s.notna().mean())
print("4. Alunos com contexto por partição: "
      f"{', '.join(f'{p} {c:.1f}%' for p, c in cobertura.items())}")

print("=" * 58)
if vazados or sem_particao:
    raise RuntimeError("Partição reprovada nas verificações acima.")
print("Partição aprovada: nenhum município atravessa as fronteiras.")

Verificações da partição
1. Municípios em mais de uma partição: 0  (esperado: 0)
2. Observações sem partição: 0  (esperado: 0)
3. Taxa de alfabetização por partição: teste 60.8%, treino 59.4%, validacao 60.0%
   amplitude entre partições: 1.4 pp (diferenças pequenas são esperadas: a partição é por município)
4. Alunos com contexto por partição: teste 90.7%, treino 90.2%, validacao 85.2%
Partição aprovada: nenhum município atravessa as fronteiras.


## 7. Gravação da tabela analítica

**Passos desta seção:** (7.1) selecionar as colunas finais e gravar a tabela no data lake; (7.2) reconciliar a gravação e registrar o dicionário de variáveis.

🎓 **Conceito, onde vive a tabela analítica:** o repositório versiona código, e os dados vivem no lake, princípio herdado da fase anterior. A tabela analítica é gravada em uma área própria (`ml/`), separada das camadas do medalhão, porque não é um produto de dados de negócio, e sim um artefato de modelagem, com público e ciclo de vida próprios. As etapas seguintes leem essa tabela, e não voltam às camadas originais.

📌 **O que a tabela carrega, e por quê:**

| Grupo | Colunas | Papel |
|---|---|---|
| Chaves | `ano`, `id_municipio` | rastreabilidade e agregação dos resultados |
| Explicativas | `rede_nome` e as nove variáveis municipais | entram no modelo |
| Resposta | `alvo` | o que se quer prever |
| Ponderação | `peso_aluno` | disponível para o treinamento ponderado, não é atributo |
| Partição | `particao` | preserva a separação entre treino, validação e teste em todas as etapas |

In [99]:
# --- 7.1 Selecionar as colunas finais e gravar no lake ---
from datetime import datetime, timezone

COLUNAS_ABT = CHAVES + FEATURES + [RESPOSTA, "peso_aluno", "particao"]
df_abt = abt[COLUNAS_ABT].copy()

print(f"Tabela analítica: {len(df_abt):,} linhas x {len(df_abt.columns)} colunas")
print(f"  chaves: {CHAVES}   resposta: {RESPOSTA}")
for familia, variaveis in FAMILIAS.items():
    print(f"  {familia:<26} {len(variaveis):>2} variáveis")
print()

garantir_credencial()
momento = datetime.now(timezone.utc)
df_abt["_processing_timestamp"] = momento.isoformat()
destino = (f"gs://{BUCKET_LAKE}/ml/abt_alfabetizacao/"
           f"data_processamento={momento:%Y-%m-%d}/abt_alfabetizacao.parquet")
df_abt.to_parquet(destino, index=False,
                  storage_options={"token": credenciais})
print(f"Gravado em: {destino}")

Tabela analítica: 1,851,852 linhas x 53 colunas
  chaves: ['ano', 'id_municipio']   resposta: alvo
  aluno                       1 variáveis
  desempenho anterior        11 variáveis
  território                  2 variáveis
  porte e oferta              6 variáveis
  infraestrutura escolar     10 variáveis
  profissionais de apoio      3 variáveis
  contexto socioeconômico    15 variáveis

Gravado em: gs://tech-challenge-fase2-lake-rm373453/ml/abt_alfabetizacao/data_processamento=2026-09-06/abt_alfabetizacao.parquet


In [102]:
# --- 7.2 Reconciliar a gravação e publicar o dicionário de dados ---
garantir_credencial()  # sessões longas: o token expira em cerca de uma hora
relido = pd.read_parquet(destino, storage_options={"token": credenciais})
status = "OK" if len(relido) == len(df_abt) else "DIVERGIU"
print(f"Reconciliação: gravado {len(df_abt):,} | relido {len(relido):,}  {status}")
print()

# Metadados de cada variável: bloco temático, fonte, referência temporal,
# natureza (medida direta, derivada de outras, ou aproximação de um conceito
# que não se mede diretamente) e a fórmula, quando houver.
METADADOS_VARIAVEIS = [
    # (variavel, bloco, fonte, referencia, natureza, descricao, formula)
    ("ano", "chave", "Silver, camada de alunos", str(CICLO_ALVO), "direta",
     "ciclo da avaliação", ""),
    ("id_municipio", "chave", "Silver, camada de alunos", str(CICLO_ALVO), "direta",
     "código IBGE do município", ""),
    ("alvo", "resposta", "Silver, camada de alunos", str(CICLO_ALVO), "derivada",
     "1 se o aluno foi classificado como alfabetizado", "proficiência >= 743"),
    ("peso_aluno", "ponderação", "Silver, camada de alunos", str(CICLO_ALVO), "direta",
     "peso amostral calibrado pelo INEP, reservado à ponderação", ""),
    ("particao", "controle", "construída neste notebook", str(CICLO_ALVO), "derivada",
     "conjunto de destino: treino, validação ou teste", "sorteio por município"),

    ("rede_nome", "aluno", "Silver, camada de alunos", str(CICLO_ALVO), "direta",
     "rede de ensino que atende o aluno", ""),

    ("rede_taxa_ant", "desempenho anterior", "Silver, indicador municipal por rede",
     str(CICLO_ANTERIOR), "direta", "taxa de alfabetização da rede do aluno no município", ""),
    ("rede_media_portugues_ant", "desempenho anterior", "Silver, indicador municipal por rede",
     str(CICLO_ANTERIOR), "direta", "proficiência média em português da rede", ""),
    ("uf_rede_taxa_ant", "desempenho anterior", "Silver, indicador municipal por rede",
     str(CICLO_ANTERIOR), "derivada", "benchmark: mediana da mesma rede entre os municípios da UF",
     "mediana por UF e rede"),
    ("rede_vs_uf", "desempenho anterior", "Silver, indicador municipal por rede",
     str(CICLO_ANTERIOR), "derivada", "posição da rede diante do padrão do seu estado",
     "rede_taxa_ant - uf_rede_taxa_ant"),
    ("rede_vs_municipio", "desempenho anterior", "Silver, indicador municipal por rede",
     str(CICLO_ANTERIOR), "derivada", "posição da rede diante do seu município",
     "rede_taxa_ant - mun_taxa_ant"),
    ("mun_taxa_ant", "desempenho anterior", "Gold, indicador municipal",
     str(CICLO_ANTERIOR), "direta", "taxa de alfabetização do município na rede pública", ""),
    ("mun_participacao_ant", "desempenho anterior", "Gold, indicador municipal",
     str(CICLO_ANTERIOR), "direta", "percentual de alunos presentes na avaliação anterior", ""),
    ("mun_taxa_ajustada_ant", "desempenho anterior", "Gold, indicador municipal",
     str(CICLO_ANTERIOR), "derivada", "taxa com ausentes contados como não alfabetizados",
     "mun_taxa_ant * mun_participacao_ant / 100"),
    ("mun_alunos_ant", "desempenho anterior", "Gold, indicador municipal",
     str(CICLO_ANTERIOR), "direta", "alunos presentes no município no ciclo anterior", ""),
    ("mun_meta_ciclo", "desempenho anterior", "Gold, metas pactuadas",
     f"{CICLO_ALVO}, pactuada previamente", "direta",
     "meta de alfabetização pactuada para o ciclo corrente", ""),
    ("mun_gap_meta", "desempenho anterior", "Gold, metas pactuadas",
     f"{CICLO_ANTERIOR} e {CICLO_ALVO}", "derivada", "esforço requerido para atingir a meta",
     "mun_meta_ciclo - mun_taxa_ant"),

    ("sigla_uf", "território", "IBGE, diretório de municípios", "estável", "direta",
     "unidade da federação", ""),
    ("nome_regiao", "território", "IBGE, diretório de municípios", "estável", "direta",
     "região do país", ""),

    ("rede_porte_atual", "porte e oferta", "Silver, camada de alunos",
     f"{CICLO_ALVO}, cadastro prévio", "derivada",
     "alunos avaliáveis na rede do aluno no município", "contagem de matriculados"),
    ("mun_porte_atual", "porte e oferta", "Silver, camada de alunos",
     f"{CICLO_ALVO}, cadastro prévio", "derivada",
     "alunos avaliáveis no município", "contagem de matriculados"),
    ("rede_peso_no_municipio", "porte e oferta", "Silver, camada de alunos",
     str(CICLO_ALVO), "derivada", "fração dos alunos do município atendida pela rede",
     "rede_porte_atual / mun_porte_atual"),
    ("esc_quantidade", "porte e oferta", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas da rede no município", ""),
    ("esc_salas_por_aluno", "porte e oferta", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "proxy", "densidade física: espaço disponível por aluno, na falta de metragem",
     "salas em uso / rede_porte_atual"),
    ("turma_media_alunos", "porte e oferta", "INEP, Censo Escolar, turmas",
     str(CICLO_ANTERIOR), "direta", "tamanho médio da turma do 2º ano, a série avaliada", ""),

    ("esc_pct_rural", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "percentual de escolas da rede em zona rural", ""),
    ("esc_pct_agua_rede", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com abastecimento de água pela rede pública", ""),
    ("esc_pct_esgoto_rede", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com esgotamento sanitário pela rede pública", ""),
    ("esc_pct_energia_rede", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com energia elétrica da rede pública", ""),
    ("esc_pct_internet", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com acesso à internet", ""),
    ("esc_pct_biblioteca", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com biblioteca", ""),
    ("esc_pct_lab_informatica", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com laboratório de informática", ""),
    ("esc_pct_quadra", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com quadra de esportes", ""),
    ("esc_pct_alimentacao", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas que oferecem alimentação aos alunos", ""),
    ("esc_pct_transporte", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "proxy", "dependência de transporte escolar, aproximação da distância entre aluno e escola",
     "alunos transportados / rede_porte_atual"),

    ("esc_pct_coordenador", "profissionais", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com coordenador pedagógico", ""),
    ("esc_pct_psicologo", "profissionais", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com psicólogo", ""),
    ("esc_pct_assistente_social", "profissionais", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com assistente social", ""),

    ("mun_pib_per_capita", "socioeconômico", "IBGE, PIB municipal", str(ANO_PIB),
     "derivada", "riqueza produzida por habitante", "PIB / população"),
    ("mun_pct_agropecuaria", "socioeconômico", "IBGE, PIB municipal", str(ANO_PIB),
     "proxy", "peso da agropecuária, aproximação do caráter rural da economia",
     "valor adicionado agropecuário / valor adicionado total"),
    ("mun_pct_servicos", "socioeconômico", "IBGE, PIB municipal", str(ANO_PIB),
     "proxy", "peso dos serviços, aproximação do caráter urbano da economia",
     "valor adicionado de serviços / valor adicionado total"),
    ("mun_populacao", "socioeconômico", "IBGE, estimativa populacional", str(ANO_PIB),
     "direta", "população do município", ""),
    ("mun_idhm", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano", "2010",
     "direta", "índice de desenvolvimento humano municipal", ""),
    ("mun_idhm_educacao", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano", "2010",
     "direta", "dimensão educação do IDHM", ""),
    ("mun_idhm_renda", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano", "2010",
     "direta", "dimensão renda do IDHM", ""),
    ("mun_renda_per_capita", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano", "2010",
     "direta", "renda domiciliar por habitante", ""),
    ("mun_gini", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano", "2010",
     "direta", "desigualdade na distribuição da renda", ""),
    ("mun_analfabetismo_adulto", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano",
     "2010", "proxy", "analfabetismo entre adultos, aproximação do capital cultural do domicílio", ""),
    ("mun_expectativa_estudo", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano",
     "2010", "direta", "expectativa de anos de estudo", ""),
    ("mun_ivs", "socioeconômico", "IPEA, Atlas da Violência", "2010", "direta",
     "índice de vulnerabilidade social", ""),
    ("mun_ivs_infraestrutura", "socioeconômico", "IPEA, Atlas da Violência", "2010",
     "direta", "vulnerabilidade de infraestrutura urbana", ""),
    ("mun_ivs_capital_humano", "socioeconômico", "IPEA, Atlas da Violência", "2010",
     "direta", "vulnerabilidade de capital humano", ""),
    ("mun_taxa_homicidio", "socioeconômico", "DataSUS, Sistema de Informação sobre Mortalidade",
     str(ANO_VIOLENCIA), "proxy", "óbitos por agressão por cem mil habitantes, aproximação da exposição à violência",
     "óbitos CID X85 a Y09 * 100000 / população"),
]

COLUNAS_META = ["variavel", "bloco", "fonte", "referencia", "natureza",
                "descricao", "formula"]
dicionario = pd.DataFrame(METADADOS_VARIAVEIS, columns=COLUNAS_META)

# Trava: o dicionário e a tabela precisam descrever exatamente o mesmo conjunto
METADADOS_GRAVACAO = ["_processing_timestamp", "data_processamento"]
na_tabela = [c for c in df_abt.columns if c not in METADADOS_GRAVACAO]
sem_descricao = [v for v in na_tabela if v not in set(dicionario["variavel"])]
orfas = [v for v in dicionario["variavel"] if v not in na_tabela]
if sem_descricao or orfas:
    raise RuntimeError(
        f"Dicionário desatualizado. Sem descrição: {sem_descricao}. "
        f"Descrições sem variável: {orfas}. Reexecute as seções 3 a 5."
    )

# Características observadas no dado
dicionario["tipo"] = [str(df_abt[v].dtype) for v in dicionario["variavel"]]
dicionario["preenchimento"] = [
    round(100 * df_abt[v].notna().mean(), 1) for v in dicionario["variavel"]]
dicionario["distintos"] = [df_abt[v].nunique() for v in dicionario["variavel"]]

dicionario.to_csv("../reports/dicionario_dados.csv", index=False,
                  encoding="utf-8")
print(f"{len(dicionario)} variáveis descritas · também salvo em "
      "reports/dicionario_dados.csv")

# O DataFrame fica disponível para inspeção e filtro no próprio notebook.
# Exemplos: dicionario.query("natureza == 'proxy'")
#           dicionario.query("bloco == 'infraestrutura'")
dicionario

Reconciliação: gravado 1,851,852 | relido 1,851,852  OK

53 variáveis descritas · também salvo em reports/dicionario_dados.csv


,variavel,bloco,fonte,referencia,natureza,descricao,formula,tipo,preenchimento,distintos
0,ano,chave,"Silver, camada de alunos",2024,direta,ciclo da avaliação,,Int64,100.0,1
1,id_municipio,chave,"Silver, camada de alunos",2024,direta,código IBGE do município,,str,100.0,5517
2,alvo,resposta,"Silver, camada de alunos",2024,derivada,1 se o aluno foi classificado como alfabetizado,proficiência >= 743,int64,100.0,2
3,peso_aluno,ponderação,"Silver, camada de alunos",2024,direta,"peso amostral calibrado pelo INEP, reservado à...",,float64,100.0,540
4,particao,controle,construída neste notebook,2024,derivada,"conjunto de destino: treino, validação ou teste",sorteio por município,str,100.0,3
5,rede_nome,aluno,"Silver, camada de alunos",2024,direta,rede de ensino que atende o aluno,,str,100.0,3
6,rede_taxa_ant,desempenho anterior,"Silver, indicador municipal por rede",2023,direta,taxa de alfabetização da rede do aluno no muni...,,float64,98.1,4059
7,rede_media_portugues_ant,desempenho anterior,"Silver, indicador municipal por rede",2023,direta,proficiência média em português da rede,,float64,98.1,6397
8,uf_rede_taxa_ant,desempenho anterior,"Silver, indicador municipal por rede",2023,derivada,benchmark: mediana da mesma rede entre os muni...,mediana por UF e rede,float64,98.1,46
9,rede_vs_uf,desempenho anterior,"Silver, indicador municipal por rede",2023,derivada,posição da rede diante do padrão do seu estado,rede_taxa_ant - uf_rede_taxa_ant,float64,98.1,5245
